In [ ]:
# log파일 추가 시 한번만 실행
import _common
_common.read_logs()

Reading logs/Huam415-1/251016_SM-S938N_TEST06_n-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST31_-M2_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST07_-M2_Fav.xlsx
Reading logs/Huam415-1/251016_SM-S938N_TEST02_n28-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST09_-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST30_-M2_Fav.xlsx
Reading logs/Huam415-1/251017_SM-S938N_TEST02_N26-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST13_-M2_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST14_-M2_Fav.xlsx
Reading logs/Huam415-1/251016_SM-S938N_TEST05_n-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST01_-M2_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST06_-M2_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST08_-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST02_-M1_Fav.xlsx
Reading logs/Huam415-1/251022_SM-S938N_TEST05_-M1_Fav.xlsx
Reading logs/Huam415-1/251017_SM-S938N_TEST03_N26-M1_Fav.xlsx
Reading logs/Huam415-1/251016_SM-S938N_TEST04

In [1]:
import os, _common, _map, _plot, _trend
df = _common.analyze_kpi(
    fname = os.path.join("logs",f"SM-S938N_All.xlsx"),
    date_list = [
        # "251016",
        # "251017",
        # "251020",
        # "251021",
        "251022", # pdcch dynamic 적용
        "251023",
    ],
)
out_dir = os.path.join("results", "All")
os.makedirs(out_dir, exist_ok=True)

_map.map_pct(df=df, out_dir=out_dir, grid_size=25, rb_min=48, sample_min=10)
_map.map_db(df=df, out_dir=out_dir, grid_size=25, rb_min=48, sample_min=10)
_map.map_coverage(df=df, out_dir=out_dir, grid_size=25, rb_min=48, sample_min=10)
_trend.kpi_each_test(df=df, out_dir=out_dir, grid_size=25, rb_min=48, sample_min=10)
_trend.rb_each_test(df=df, out_dir=out_dir, rb_min=0)

site_list = [
    'Huam415-1',
    'Huam345-5',
    'Namsan',
]
for site in site_list:
    df_site = df[df["route"]==site].copy()
    
    out_dir = os.path.join("results", site)
    os.makedirs(out_dir, exist_ok=True)
    
    _map.map_pct(df=df_site, out_dir=out_dir, grid_size=5, rb_min=48, sample_min=2)
    _trend.kpi_each_test(df=df_site, out_dir=out_dir, grid_size=5, rb_min=48, sample_min=2)

Reading logs/SM-S938N_All.xlsx
✅ Read Complete : 62029 lines
✅ Saved: results/All/map_25m_DL_Tput_ratio.html (rows=190)
✅ Saved: results/All/map_25m_RSRP_diff.html (rows=190)
✅ Saved: results/All/map_25m_RSRP_n28.html (rows=190)
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_01_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_02_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_03_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_04_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_05_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_06_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_07_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_08_Huam415-1.html
Saved HTML: results/All/plot_kpi_each_test/251022/Huam415-1/251022_09_Huam415-1.html
Saved 

In [ ]:
# _plot.plot_kpi(df=df, grid_size=1, out_dir=out_dir, title=title)
# _plot.plot_tput_vs_sinr(df=df, grid_size=1, out_dir=out_dir, title=title)

In [ ]:
import os, _common, _map, _plot, _trend
import numpy as np
import pandas as pd

df = _common.analyze_kpi(
    fname = os.path.join("logs",f"SM-S938N_All.xlsx"),
    date_list = [], # all
    rb_min=48
)
display(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

bins = np.arange(df["RSRP"].min(), df["RSRP"].max() + 1, 1)
df["RSRP_bin"] = pd.cut(df["RSRP"], bins=bins, right=False)
df["RSRP_bin_center"] = df["RSRP_bin"].apply(lambda x: (x.left + x.right) / 2)

df_grouped = (
    df.groupby(["Band", "RSRP_bin_center"], observed=True)["DL_Tput"]
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
for band, color in zip(["n26", "n28"], ["blue", "red"]):
    df_sub = df_grouped[df_grouped["Band"] == band]
    ax.plot(
        df_sub["RSRP_bin_center"], df_sub["DL_Tput"],
        marker="o", markersize=3, label=band, color=color, linewidth=1, alpha=0.85
    )
ax.set_xlim(-120, -40)
ax.set_xlabel("RSRP [dBm]")
ax.set_ylabel("DL Throughput [Mbps]")
ax.legend()

ax.grid(True, which='major', linestyle='-', linewidth=0.6, alpha=0.6)
ax.minorticks_on()
ax.grid(True, which='minor', linestyle=':', linewidth=0.4, alpha=0.4)

ax.invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
df_pair = _common.grid_kpi(df, grid_size=grid_size)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ----------------------------
# 거리 구간 정의 (200m step)
# ----------------------------
bins = [0, 200, 400, 600, 800, 1000, np.inf]
labels = [
    "0–200 m", "200–400 m", "400–600 m",
    "600–800 m", "800–1000 m", "1000 m+"
]
df_pair["distance_bin"] = pd.cut(df_pair["distance"], bins=bins, labels=labels, right=False)

# ----------------------------
# 색상 로직
# ----------------------------
source_order = labels
base_sources = [s for s in source_order if s not in ['clean']]
num_base = len(base_sources)

cmap = matplotlib.colormaps.get_cmap("tab20b")
base_colors = [cmap(i / 20) for i in np.linspace(4, 19, num=num_base)]

# ----------------------------
# 그래프 그리기
# ----------------------------
plt.figure(figsize=(15,7))
plt.title("Average DL Throughput vs RSRP (n28)", fontsize=13)
plt.xlabel("RSRP_n28 (dBm)")
plt.ylabel("Average DL_Tput_n28 (Mbps)")
plt.grid(True, linestyle="--", alpha=0.4)

# 거리 순서를 역순으로 (멀리 → 가까이)
for label, color in zip(reversed(base_sources), reversed(base_colors)):
    df_sub = df_pair[df_pair["distance_bin"] == label].copy()
    if df_sub.empty:
        continue

    # 1 dB 단위 binning
    df_sub.loc[:, "RSRP_bin"] = df_sub["RSRP_n28"].astype(int)

    # 평균 계산
    df_binned = (
        df_sub.groupby("RSRP_bin", observed=True)["DL_Tput_n28"]
              .mean()
              .reset_index()
              .sort_values("RSRP_bin")
    )

    plt.plot(
        df_binned["RSRP_bin"],
        df_binned["DL_Tput_n28"],
        label=label,
        color=color,
        linewidth=1,
        marker='o',
        markersize=3,
    )

# ----------------------------
# 축/범례/레이아웃 설정
# ----------------------------
plt.gca().invert_xaxis()
plt.xlim(-60, -120)
# plt.ylim(0, 70)


# ✅ 범례를 그래프 안쪽에, 역순으로 배치
handles, legend_labels = plt.gca().get_legend_handles_labels()
plt.legend(
    handles[::-1],
    legend_labels[::-1],
    title="Distance",
    fontsize=11,
    title_fontsize=11,
    loc="upper right",
    frameon=True,
    framealpha=0.9,
    facecolor="white",
    edgecolor="gray"
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ----------------------------
# uhd_max 구간 정의 (3 dB step)
# ----------------------------
uhd_min, uhd_max = -38, df_pair["uhd_max"].max()
uhd_bins = np.arange(uhd_min, np.ceil(uhd_max) + 3, 3)
uhd_labels = [f"{uhd_bins[i]}~{uhd_bins[i+1]} dB" for i in range(len(uhd_bins)-1)]

# -38 이하 제외 후 구간화
df_pair_filtered = df_pair[df_pair["uhd_max"] > -38].copy()
df_pair_filtered["uhd_bin"] = pd.cut(df_pair_filtered["uhd_max"], bins=uhd_bins, labels=uhd_labels, right=False)

# ----------------------------
# 색상 로직
# ----------------------------
source_order = uhd_labels
base_sources = [s for s in source_order if s not in ['clean']]
num_base = len(base_sources)

cmap = matplotlib.colormaps.get_cmap("tab20b")
base_colors = [cmap(i / 20) for i in np.linspace(4, 19, num=num_base)]

# ----------------------------
# 그래프 그리기
# ----------------------------
plt.figure(figsize=(15,7))
plt.title("Average DL Throughput vs RSRP (n28) by UHD_MAX bins", fontsize=13)
plt.xlabel("RSRP_n28 (dBm)")
plt.ylabel("Average DL_Tput_n28 (Mbps)")
plt.grid(True, linestyle="--", alpha=0.4)

# 구간 역순으로 (큰 UHD → 작은 UHD)
for label, color in zip(reversed(base_sources), reversed(base_colors)):
    df_sub = df_pair_filtered[df_pair_filtered["uhd_bin"] == label].copy()
    if df_sub.empty:
        continue

    # 1 dB 단위 binning
    df_sub.loc[:, "RSRP_bin"] = df_sub["RSRP_n28"].astype(int)

    # 평균 계산
    df_binned = (
        df_sub.groupby("RSRP_bin", observed=True)["DL_Tput_n28"]
              .mean()
              .reset_index()
              .sort_values("RSRP_bin")
    )

    plt.plot(
        df_binned["RSRP_bin"],
        df_binned["DL_Tput_n28"],
        label=label,
        color=color,
        linewidth=1,
        marker='o',
        markersize=3,
    )

# ----------------------------
# 축/범례/레이아웃 설정
# ----------------------------
plt.gca().invert_xaxis()
plt.xlim(-60, -120)
# plt.ylim(0,80)

handles, legend_labels = plt.gca().get_legend_handles_labels()
plt.legend(
    handles[::-1],
    legend_labels[::-1],
    title="UHD_MAX (dB)",
    fontsize=11,
    title_fontsize=11,
    loc="upper right",
    frameon=True,
    framealpha=0.9,
    facecolor="white",
    edgecolor="gray"
)

plt.tight_layout()
plt.show()